In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import json
import plotly.graph_objects as go
import logging

import utils
import z_utils
import network

sequence_length = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


# Data Loading

In [3]:
loader = utils.get_m2s_loader(128)

# Sparate the data into train and test
train_loader = []
test_loader = []

for i, data in enumerate(loader):
    if i % 10 == 0:
        test_loader.append(data)
    else:
        train_loader.append(data)

test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')
print(test_df.shape)
print(test_df.head())


(1682533, 19)
       LCLid  energy(kWh/hh)_smoothed  energy(kWh/hh)   mean  median     std  \
0  MAC000051                   0.3409           0.247  0.323    0.21  0.3232   
1  MAC000051                   0.3760           0.318  0.323    0.21  0.3232   
2  MAC000051                   0.3652           0.284  0.323    0.21  0.3232   
3  MAC000051                   0.4082           0.306  0.323    0.21  0.3232   
4  MAC000051                   0.4033           0.337  0.323    0.21  0.3232   

   min    max  gradient  kmedoid_clusters  spike_type  spike_magnitude  \
0  0.0  3.462    0.1614                17           0              1.0   
1  0.0  3.462    0.1614                17           0              1.0   
2  0.0  3.462    0.1614                17           0              1.0   
3  0.0  3.462    0.1614                17           0              1.0   
4  0.0  3.462    0.1614                17           0              1.0   

   temperature  windSpeed  humidity  date_sin  date_cos  tim

# Simple Wasserstein GAN

In [4]:
class Generator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, hidden_size):
        super(Generator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )

        self.fc = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )
    
    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude), dim=-1)

        return self.fc(out)

# We will put all the auxiliary information for the Discriminator too, to make it better.
class Discriminator(nn.Module):
    def __init__(self, s_embed_size, c_embed_size, time_size, time_out, stat_size, stat_out, fc1_hidden_size, fc1_out, fc2_hidden_size):
        super(Discriminator, self).__init__()

        self.Embedding_spike = nn.Embedding(5, s_embed_size)
        self.Embedding_cluster = nn.Embedding(20, c_embed_size)

        self.time_net = nn.Sequential(
            nn.Linear(4, time_size),
            nn.ReLU(),
            nn.Linear(time_size, time_out),
            nn.ReLU()
        )

        self.statistical_net = nn.Sequential(
            nn.Linear(6, stat_size),
            nn.ReLU(),
            nn.Linear(stat_size, stat_out),
            nn.ReLU()
        )
        
        self.fc1 = nn.Sequential(
            nn.Linear(s_embed_size + c_embed_size + time_out + stat_out + 3 + 6 + 1 + 1, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_hidden_size),
            nn.ReLU(),
            nn.Linear(fc1_hidden_size, fc1_out),
            nn.ReLU()
        )

        self.fc2 = nn.Sequential(
            nn.Linear(fc1_out + 1, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, fc2_hidden_size),
            nn.ReLU(),
            nn.Linear(fc2_hidden_size, 1)
        )

    def forward(self, weather, cluster, time, statistical, spike_type, spike_magnitude, energy):
        sp = self.Embedding_spike(spike_type).squeeze(2)
        c = self.Embedding_cluster(cluster).squeeze(2)
        t = self.time_net(time)
        s = self.statistical_net(statistical)
        out = torch.cat((weather, c, t, s, statistical, sp, spike_magnitude, energy), dim=-1)
        out = self.fc1(out)
        out = torch.cat((out, energy), dim=-1)
        return self.fc2(out)

# Parameter Selection

In [5]:
gen_parameter_range = {
    's_embed_size': [2, 3, 4],
    'c_embed_size': [2, 3, 4],
    'time_size': [32, 64, 128, 256],
    'time_out': [8, 16, 32],
    'stat_size': [32, 64, 128, 256],
    'stat_out': [16, 32, 64],
    'hidden_size': [64, 128, 256, 512]
}

dis_parameter_range = {
    's_embed_size': [2, 3, 4],
    'c_embed_size': [2, 3, 4],
    'time_size': [32, 64, 128, 256],
    'time_out': [8, 16, 32],
    'stat_size': [32, 64, 128, 256],
    'stat_out': [16, 32, 64],
    'fc1_hidden_size': [64, 128, 256],
    'fc1_out': [4, 8, 16, 32],
    'fc2_hidden_size': [64, 128, 256, 512]
}

In [6]:
gen = Generator(2, 2, 32, 8, 32, 16, 64).to(device)
disc = Discriminator(2, 2, 32, 8, 32, 16, 64, 4, 64).to(device)

logging.basicConfig(filename='WGAN_hp_search.log', level=logging.INFO)
best_gen_config, best_dis_config = z_utils.WGAN_hp_search(gen_parameter_range, dis_parameter_range, train_loader, test_loader, gen, disc, device)


2024-07-11 15:18:31,444 - INFO - Optimizing Generator parameter s_embed_size
2024-07-11 15:18:31,445 - INFO - Current Generator configuration: {'s_embed_size': 2, 'c_embed_size': 2, 'time_size': 32, 'time_out': 8, 'stat_size': 32, 'stat_out': 16, 'hidden_size': 64}
2024-07-11 15:18:31,567 - INFO - Optimizing Discriminator parameter s_embed_size with Generator parameter s_embed_size = 2
2024-07-11 15:18:31,567 - INFO - Optimizing Discriminator Parameter s_embed_size with value 2
2024-07-11 15:18:52,106 - INFO - Epoch 0, Discriminator Loss: -0.019402365895017745, Generator Loss: -0.10813875914532312
2024-07-11 15:19:14,601 - INFO - Epoch 1, Discriminator Loss: -0.03120191729165056, Generator Loss: 0.010815107305828521
2024-07-11 15:19:37,642 - INFO - Epoch 2, Discriminator Loss: -0.03549799236074056, Generator Loss: 0.2264536478284265
2024-07-11 15:20:01,943 - INFO - Epoch 3, Discriminator Loss: -0.02888829832106761, Generator Loss: 0.11019652241697353
2024-07-11 15:20:24,476 - INFO - Ep

Best Generator configuration: {'s_embed_size': 3, 'c_embed_size': 2, 'time_size': 32, 'time_out': 16, 'stat_size': 32, 'stat_out': 16, 'hidden_size': 64}
Best Discriminator configuration: {'s_embed_size': 3, 'c_embed_size': 2, 'time_size': 32, 'time_out': 8, 'stat_size': 32, 'stat_out': 16, 'fc1_hidden_size': 64, 'fc1_out': 4, 'fc2_hidden_size': 512}

In [7]:
best_gen_config

{'s_embed_size': 3,
 'c_embed_size': 2,
 'time_size': 32,
 'time_out': 16,
 'stat_size': 32,
 'stat_out': 16,
 'hidden_size': 64}

In [8]:
best_dis_config

{'s_embed_size': 3,
 'c_embed_size': 2,
 'time_size': 32,
 'time_out': 8,
 'stat_size': 32,
 'stat_out': 16,
 'fc1_hidden_size': 64,
 'fc1_out': 4,
 'fc2_hidden_size': 512}

# Training

In [4]:
# Configure logging
logging.basicConfig(filename='WGAN_training.log', level=logging.INFO)

# Use the best hyperparameters obtained from the search
best_gen_config_path = 'Config/wgan_g_config.json'
with open(best_gen_config_path) as f:
    best_gen_config = json.load(f)

best_dis_config_path = 'Config/wgan_d_config.json'
with open(best_dis_config_path) as f:
    best_dis_config = json.load(f)

gen = Generator(**best_gen_config).to(device)
dis = Discriminator(**best_dis_config).to(device)

trained_gen, trained_dis = z_utils.train_WGAN(gen, dis, train_loader, test_loader, device, num_epochs=100, clip_value=0.06)

torch.save(trained_gen.state_dict(), '../../Models/WGAN_generator.pt')
torch.save(trained_dis.state_dict(), '../../Models/WGAN_discriminator.pt')

2024-07-14 16:50:54,292 - INFO - Iteration 999, Discriminator Loss: -0.2199704945087433, Generator Loss: 0.6421157121658325
2024-07-14 16:50:59,139 - INFO - Iteration 1999, Discriminator Loss: 0.07612341642379761, Generator Loss: 0.1444093883037567
2024-07-14 16:51:03,911 - INFO - Iteration 2999, Discriminator Loss: -0.03739026188850403, Generator Loss: -0.4101566672325134
2024-07-14 16:51:09,071 - INFO - Epoch 0, Discriminator Loss: -0.10073138418651763, Generator Loss: -0.29244528531757885
2024-07-14 16:51:13,804 - INFO - Iteration 999, Discriminator Loss: -0.051225632429122925, Generator Loss: -0.30483943223953247
2024-07-14 16:51:18,617 - INFO - Iteration 1999, Discriminator Loss: -0.11470814049243927, Generator Loss: -0.032040588557720184
2024-07-14 16:51:23,392 - INFO - Iteration 2999, Discriminator Loss: -0.10750212520360947, Generator Loss: -0.055566154420375824
2024-07-14 16:51:28,599 - INFO - Epoch 1, Discriminator Loss: -0.05944560924354864, Generator Loss: -0.22660771296137

New Best Model


2024-07-14 16:55:07,254 - INFO - Iteration 999, Discriminator Loss: -0.2054285705089569, Generator Loss: -0.48572462797164917
2024-07-14 16:55:12,880 - INFO - Iteration 1999, Discriminator Loss: -0.17739194631576538, Generator Loss: -0.2840198278427124
2024-07-14 16:55:18,490 - INFO - Iteration 2999, Discriminator Loss: -0.1419692039489746, Generator Loss: -0.19240963459014893
2024-07-14 16:55:24,568 - INFO - Epoch 12, Discriminator Loss: -0.19262999559746308, Generator Loss: -0.41034968427114205
2024-07-14 16:55:30,110 - INFO - Iteration 999, Discriminator Loss: -0.16887372732162476, Generator Loss: -0.22451287508010864
2024-07-14 16:55:35,617 - INFO - Iteration 1999, Discriminator Loss: -0.21373042464256287, Generator Loss: 0.10342971980571747
2024-07-14 16:55:41,085 - INFO - Iteration 2999, Discriminator Loss: -0.11155632138252258, Generator Loss: 0.574334442615509
2024-07-14 16:55:47,307 - INFO - Epoch 13, Discriminator Loss: -0.17525628141535113, Generator Loss: 0.5659383765153604

New Best Model


2024-07-14 16:56:39,367 - INFO - Iteration 999, Discriminator Loss: -0.21184706687927246, Generator Loss: -0.337926983833313
2024-07-14 16:56:45,214 - INFO - Iteration 1999, Discriminator Loss: -0.11201286315917969, Generator Loss: -0.7456450462341309
2024-07-14 16:56:50,862 - INFO - Iteration 2999, Discriminator Loss: -0.19317281246185303, Generator Loss: -0.7894315123558044
2024-07-14 16:56:57,060 - INFO - Epoch 16, Discriminator Loss: -0.20193648405896833, Generator Loss: -0.7216841048664517


...

New Best Model


2024-07-14 17:24:24,203 - INFO - Iteration 999, Discriminator Loss: -0.07342332601547241, Generator Loss: 0.07279425859451294
2024-07-14 17:24:29,192 - INFO - Iteration 1999, Discriminator Loss: -0.06413212418556213, Generator Loss: 0.11389908194541931
2024-07-14 17:24:34,075 - INFO - Iteration 2999, Discriminator Loss: -0.12827423214912415, Generator Loss: 0.41166576743125916
2024-07-14 17:24:39,347 - INFO - Epoch 98, Discriminator Loss: -0.07954522389538433, Generator Loss: 0.2563859884312499


New Best Model


2024-07-14 17:24:44,207 - INFO - Iteration 999, Discriminator Loss: -0.03923596814274788, Generator Loss: 0.15662187337875366
2024-07-14 17:24:49,231 - INFO - Iteration 1999, Discriminator Loss: 0.028463438153266907, Generator Loss: 0.12367111444473267
2024-07-14 17:24:54,163 - INFO - Iteration 2999, Discriminator Loss: -0.07034122198820114, Generator Loss: 0.11023765802383423
2024-07-14 17:24:59,351 - INFO - Epoch 99, Discriminator Loss: -0.08107711329988612, Generator Loss: 0.2315005484013871
2024-07-14 17:24:59,352 - INFO - Training complete.


New Best Model


# Performance

In [5]:
best_gen_config_path = 'Config/wgan_g_config.json'
with open(best_gen_config_path) as f:
    best_gen_config = json.load(f)

best_dis_config_path = 'Config/wgan_d_config.json'
with open(best_dis_config_path) as f:
    best_dis_config = json.load(f)

gen = Generator(**best_gen_config).to(device)
dis = Discriminator(**best_dis_config).to(device)

gen.load_state_dict(torch.load('../../Models/WGAN_generator.pt'))
dis.load_state_dict(torch.load('../../Models/WGAN_discriminator.pt'))

gen.eval()
dis.eval()


Discriminator(
  (Embedding_spike): Embedding(5, 3)
  (Embedding_cluster): Embedding(20, 2)
  (time_net): Sequential(
    (0): Linear(in_features=4, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=8, bias=True)
    (3): ReLU()
  )
  (statistical_net): Sequential(
    (0): Linear(in_features=6, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
  )
  (fc1): Sequential(
    (0): Linear(in_features=40, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
    (5): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=5, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1, bias=True)
  )
)

## The Training Set

In [6]:
eva_training_loader = utils.get_m2s_loader(1)
long_fake_energy = torch.Tensor().to(device)
long_real_energy = torch.Tensor().to(device)

# For longer sequences
for i in range(12):
    weather, cluster, time, statistical, spike, real_energy = next(iter(eva_training_loader))
    
    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike[:, :, 0:1].to(dtype=torch.int32)  # or torch.long for indexing
    spike_magnitude = spike[:, :, 1:].to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)

    fake_energy = gen(weather, cluster, time, statistical, spike_type, spike_magnitude)
    fake_energy = fake_energy.to(device)

    long_fake_energy = torch.cat((long_fake_energy, fake_energy), dim=-1)
    real_energy = real_energy.to(device)
    long_real_energy = torch.cat((long_real_energy, real_energy), dim=-1)

long_fake_energy = long_fake_energy.flatten().detach().cpu().numpy()
long_real_energy = long_real_energy.flatten().detach().cpu().numpy()

# plot the results
fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_real_energy, mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, 200), y=long_fake_energy, mode='lines', name='Generated', line=dict(color='orange')))

fig.show()

## The Test Set

In [7]:
# Get the test df at Data_processed\Kmedoids_with_Weather_Espike_test.csv
test_df = pd.read_csv('../../Data_processed/Kmedoids_with_Weather_Espike_test.csv')
weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(48 * 3, test_df)

weather = weather.to(device)
cluster = cluster.to(device).int()
time = time.to(device)
statistical = statistical.to(device)
spike_type = spike_type.to(dtype=torch.int32)  # or torch.long for indexing
spike_magnitude = spike_magnitude.to(dtype=torch.float32)
spike_type = spike_type.to(device)
spike_magnitude = spike_magnitude.to(device)

fake_energy = gen(weather, cluster, time, statistical, spike_type, spike_magnitude).cpu().detach().numpy().squeeze(0)
real_energy = real_energy

print(fake_energy.shape)
print(real_energy.shape)

fig = go.Figure()

fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=real_energy[:, 0], mode='lines', name='Real', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=np.arange(0, real_energy.shape[0]), y=fake_energy[:, 0], mode='lines', name='Generated', line=dict(color='orange')))
fig.show()


(144, 1)
(144, 1)


# Evaluation

Similarity and Correlation with the real energy profile

In [7]:
mse = []
srcc = []
corr = []

for i in range(100):
    weather, cluster, time, statistical, spike_type, spike_magnitude, noise, real_energy = utils.get_information(48 * 3, test_df)

    weather = weather.to(device)
    cluster = cluster.to(device).int()
    time = time.to(device)
    statistical = statistical.to(device)
    spike_type = spike_type.to(dtype=torch.int32)  # or torch.long for indexing
    spike_magnitude = spike_magnitude.to(dtype=torch.float32)
    spike_type = spike_type.to(device)
    spike_magnitude = spike_magnitude.to(device)

    synthetic_profile = gen(weather, cluster, time, statistical, spike_type, spike_magnitude).flatten().detach().cpu().numpy()
    real_energy = real_energy.flatten()

    mse.append(utils.mean_squared_error(synthetic_profile, real_energy))
    srcc.append(utils.spearman_correlation(synthetic_profile, real_energy))
    corr.append(utils.pearson_correlation(synthetic_profile, real_energy))

print(f'MSE: {np.mean(mse)}')
print(f'SRCC: {np.mean(srcc)}')
print(f'Correlation: {np.mean(corr)}')



MSE: 0.0859257330764616
SRCC: 0.3122957149158181
Correlation: 0.5591400513783276
